6402-010302D Черных Вероника

# Введение в MapReduce модель на Python


In [2]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [3]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [4]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [5]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [6]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [7]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [8]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [9]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [10]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [11]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [12]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [13]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [14]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [15]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [16]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.9700673858507356)),
 (1, np.float64(1.9700673858507356)),
 (2, np.float64(1.9700673858507356)),
 (3, np.float64(1.9700673858507356)),
 (4, np.float64(1.9700673858507356))]

## Inverted index

In [17]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [18]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('it', 18)]),
 (1, [('a', 2), ('is', 18), ('what', 10)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.015030237649474754)),
   (None, np.float64(0.02269421470082844)),
   (None, np.float64(0.03613164686281767)),
   (None, np.float64(0.06698420658935966)),
   (None, np.float64(0.07319325891058603)),
   (None, np.float64(0.16789562041297068)),
   (None, np.float64(0.1744560845171773)),
   (None, np.float64(0.19829924600424353)),
   (None, np.float64(0.21625638428944283)),
   (None, np.float64(0.2551929385409064)),
   (None, np.float64(0.26316525549017245)),
   (None, np.float64(0.2760492193651567)),
   (None, np.float64(0.285400241555538)),
   (None, np.float64(0.2955967973713639)),
   (None, np.float64(0.3154444867716899)),
   (None, np.float64(0.319400950569656)),
   (None, np.float64(0.3320551446545623)),
   (None, np.float64(0.3775665677911082)),
   (None, np.float64(0.39639944478437394)),
   (None, np.float64(0.43452810430029276)),
   (None, np.float64(0.47330017458106266))]),
 (1,
  [(None, np.float64(0.5552385318974472)),
   (None, np.float64(0.61498164

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

In [21]:
# Необходимые библиотеки в упражнениях
import random
import numpy as np
from typing import Iterator, NamedTuple, List, Tuple, Dict
from itertools import groupby, chain
from operator import itemgetter
from collections import defaultdict

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [22]:
def MAP(num_list):
    return max(num_list)

def REDUCE(num_list):
    return max(num_list)

def RECORDREADER(count):
    return [random.randint(0, 100) for i in range(count)]


record = RECORDREADER(100)
print("LIST:",record)
parts = 5

record_partitional = np.array_split(record, len(record) // parts if len(record) % parts == 0 else len(record) // parts + 1)
record_partitional = [list(arr) for arr in record_partitional]
print("MAX:",max(map(max, record_partitional)))

LIST: [34, 65, 27, 23, 74, 65, 16, 1, 89, 37, 30, 35, 50, 41, 92, 20, 59, 7, 7, 1, 98, 70, 25, 81, 20, 51, 53, 0, 72, 86, 15, 97, 5, 38, 33, 84, 59, 11, 37, 32, 80, 64, 18, 64, 13, 49, 93, 14, 79, 16, 59, 73, 49, 52, 11, 42, 24, 90, 31, 85, 88, 83, 60, 99, 59, 91, 67, 77, 74, 96, 18, 91, 5, 71, 65, 69, 1, 3, 68, 57, 28, 17, 62, 36, 75, 39, 83, 93, 98, 89, 57, 65, 0, 13, 16, 98, 92, 91, 70, 35]
MAX: 99


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [28]:
def RECORDREADER(count):
    return [random.randint(0, 100) for _ in range(count)]

def MAP(num):
    return (1, num)

def REDUCE(_, numbers: Iterator[NamedTuple]):
    sum = 0
    item_count = 0
    for number in numbers:
        sum += number
        item_count += 1
    if item_count > 0:
        yield ('AVG', sum / item_count)
    else:
        yield ('AVG', 0)

def flatten(list_of_lists):
    for sublist in list_of_lists:
        yield from sublist

rec = RECORDREADER(100)

output = [MAP(x) for x in rec]
results = [REDUCE(1, (item[1] for item in output))]
output = list(flatten(results))
print(output)


[('AVG', 44.24)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [30]:
def group_by_key(iterable):
    return [(key, [item[1] for item in group]) for key, group in groupby(sorted(iterable, key=itemgetter(0)), key=itemgetter(0))]

def MAP(num):
    return (1, num)

def REDUCE(_, numbers: Iterator[NamedTuple]):
    sum = 0
    item_count = 0
    for number in numbers:
        sum += number
        item_count += 1
    if item_count > 0:
        yield ('AVG', sum / item_count)
    else:
        yield ('AVG', 0)

def RECORDREADER(count):
    return [random.randint(0, 100) for _ in range(count)]

def flatten(list_of_lists):
    for sublist in list_of_lists:
        yield from sublist

map_output = list(map(lambda x: MAP(x), RECORDREADER(100)))
shuffle_output = group_by_key(map_output)
print(shuffle_output)

output = list(flatten(map(lambda x: REDUCE(*x), shuffle_output)))
print(output)


[(1, [44, 40, 30, 31, 73, 85, 72, 41, 12, 19, 7, 59, 36, 21, 58, 41, 87, 24, 48, 18, 66, 79, 59, 10, 98, 2, 70, 18, 83, 100, 77, 99, 31, 55, 37, 46, 58, 81, 67, 22, 89, 74, 70, 36, 90, 89, 37, 44, 58, 23, 33, 98, 30, 7, 63, 68, 49, 97, 96, 67, 23, 16, 65, 53, 76, 72, 74, 9, 96, 32, 80, 27, 30, 23, 64, 40, 79, 65, 94, 54, 34, 14, 29, 56, 90, 66, 87, 63, 35, 26, 10, 51, 20, 1, 90, 88, 100, 74, 44, 59])]
[('AVG', 53.31)]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [32]:
reduce_count = 2

def input_data_source():
    global count
    data = ["apple", "banana", "apple", "orange", "banana", "kiwi", "kiwi", "apple", "kiwi"]

    def data_reader(chunk):
        for item in chunk:
            yield (item, None)

    chunk_size = max(1, len(data) // count)
    for index in range(0, len(data), chunk_size):
        yield data_reader(data[index:index+chunk_size])

def map_task(key, dummy_value):
    yield (key, None)

def partition_task(key):
    global reduce_count
    return hash(key) % reduce_count

def reduce_task(key, dummy_value):
    yield (key, None)


count = 2
distinct_items = MapReduceDistributed(input_data_source, map_task, reduce_task, partition_task)

distinct_items = [key for (_, partition) in distinct_items for (key, _) in partition]
print("Уникальные значения:", distinct_items)


9 key-value pairs were sent over a network.
Уникальные значения: ['banana', 'apple', 'kiwi', 'orange']


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [35]:
def RECORDREADER():
    dataset = [(25, 20), (15, 10), (30, 35), (40, 30), (10, 5)]
    for t in dataset:
        yield (t, None)

def MAP(t, _):
    if t[0] > t[1]:
        yield (t, t)

def REDUCE(_, values):
    for value in values:
        yield value

result = list(MapReduce(RECORDREADER, MAP, REDUCE))

print("Кортежи, где 1-й элемент больше 2-го:", [item[0] for item in result])

Кортежи, где 1-й элемент больше 2-го: [25, 15, 40, 10]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [46]:
S = {10, 5, 15}

def MAP(t, _):
    """Фильтруем атрибуты"""
    filtered_result = dict(filter(lambda item: item[0] in S, t.items()))
    key = tuple(sorted(filtered_result.items()))
    yield (key, key)


def REDUCE(_, values):
    """Удаляем повторяющиеся значения"""
    yield values[0]

def RECORDREADER():
    """Генератор данных."""
    yield ({1: "Paris", 2: "London", 5: "Berlin"}, None)
    yield ({1: "France", 3: "Italy", 10: "Germany"}, None)
    yield ({5: "Europe", 7: "Asia", 10: "Africa"}, None)

result = list(MapReduce(RECORDREADER, MAP, REDUCE))

print("Проекция:", result)

Проекция: [((5, 'Berlin'),), ((10, 'Germany'),), ((5, 'Europe'), (10, 'Africa'))]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [47]:
def MAP(t, _):
    """Каждый входной элемент превращается в пару (t, t)"""
    yield (t, t)

def REDUCE(t, values):
    """Удаляет дубликаты и возвращает (t, t)"""
    yield (t, t)

def RECORDREADER():
    """Объединяем два множества"""
    set1 = [(10, None), (15, None), (30, None)]
    set2 = [(20, None), (30, None), (45, None)]
    return list(chain(set1, set2))


result = list(MapReduce(RECORDREADER, MAP, REDUCE))

print("Объединение: ", result)


Объединение:  [(10, 10), (15, 15), (30, 30), (20, 20), (45, 45)]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [49]:
def MAP(t, _):
    """Каждый элемент превращается в пару (t, t)"""
    yield (t, t)

def REDUCE(t, values):
    """ Создаем пару (t, t) по условию"""
    values_list = list(values)
    if len(values_list) == 2 and values_list[0] == t and values_list[1] == t:
        yield (t, t)

def RECORDREADER():
    """Пересечение двух множеств"""
    set1 = [10, 15, 30, 45]
    set2 = [20, 30, 50, 75]
    return [(t, None) for t in set1] + [(t, None) for t in set2]

result = list(MapReduce(RECORDREADER, MAP, REDUCE))

print("Пересечение: ", result)


Пересечение:  [(30, 30)]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [55]:
def MAP(t, source):
    """Помечаем каждый элемент его источником (1 для R и 0 для S)"""
    yield (t, source)

def REDUCE(t, sources):
    """Оставляем только элементы, которые есть в R, но нет в S"""
    # Если встретили только метку 'R' и не встретили 'S'
    if 'R' in sources and 'S' not in sources:
        yield t

def RECORDREADER():
    """Генератор данных с двумя множествами R и S"""
    # Множество R (источник 'R')
    for item in [(1, "lion"), (2, "tiger"), (3, "leopard"), (4, "jaguar")]:
        yield (item, 'R')

    # Множество S (источник 'S')
    for item in [(2, "tiger"), (3, "leopard"), (5, "cheetah"), (6, "panther")]:
        yield (item, 'S')

result = list(MapReduce(RECORDREADER, MAP, REDUCE))

print("Разница: ")
for item in result:
    print(f"{item[0]}: {item[1]}")

Разница: 
1: lion
4: jaguar


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [57]:
def MAP(record, source):
    """Формирует пары (b, (source, value))"""
    if source == "R":
        a, b = record
        yield (b, ("R", a))
    elif source == "S":
        b, c = record
        yield (b, ("S", c))

def REDUCE(b, values):
    """Создает пары (a, b, c)"""
    r_values = [a for src, a in values if src == "R"]
    s_values = [c for src, c in values if src == "S"]

    for a in r_values:
        for c in s_values:
            yield (a, b, c)

def RECORDREADER():
    """Данные для соединения: R(a, b) и S(b, c)"""
    R = [("lion", 1), ("tiger", 2), ("leopard", 3), ("jaguar", 4)]
    S = [(1, "Northern"), (2, "African"), (3, "Indian"), (4, "European")]
    return [(r, "R") for r in R] + [(s, "S") for s in S]

result = list(MapReduce(RECORDREADER, MAP, REDUCE))

print("Natural Join: ", result)


Natural Join:  [('lion', 1, 'Northern'), ('tiger', 2, 'African'), ('leopard', 3, 'Indian'), ('jaguar', 4, 'European')]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [59]:
# Данные: (студент, course_id, название_курса)
data = [
    ("Alex", 101, "Math"),
    ("Mary", 101, "Math"),
    ("Jacob", 102, "Python"),
    ("Elena", 102, "Python"),
    ("Josh", 103, "Art"),
]

def MAP_GROUP(course_id, student, course_name):
    """Группируем студентов по ID курса"""
    yield (course_id, student)

def REDUCE_GROUP(course_id, students):
    """Формируем информацию о количестве студентов на курсе"""
    yield f"On course {course_id} study {len(students)} students"

def RECORDREADER():
    """Генератор данных о записи на курсы"""
    return [(student, course_id, course_name) for student, course_id, course_name in data]

def MapReduce(reader, mapper, reducer):
    """Реализация MapReduce для группировки данных"""
    intermediate = {}

    # Этап Map
    for student, course_id, course_name in reader():
        for key, value in mapper(course_id, student, course_name):
            if key not in intermediate:
                intermediate[key] = []
            intermediate[key].append(value)

    # Этап Reduce
    for course_id, students in intermediate.items():
        yield from reducer(course_id, students)

# Запуск обработки
output = list(MapReduce(RECORDREADER, MAP_GROUP, REDUCE_GROUP))

# Вывод результатов
for line in output:
    print(line)

On course 101 study 2 students
On course 102 study 2 students
On course 103 study 1 students


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


Например, при умножении матрицы на вектор

In [ ]:
NUM_REDUCERS = 2  # Количество редьюсеров
CHUNK_SIZE = 2    # Размер чанка данных

class MatrixRow:
    """Представляет элемент матрицы с номером строки, столбца и значением."""
    def __init__(self, row, col, value):
        self.row = row
        self.col = col
        self.value = value

class VectorElement:
    """Представляет элемент вектора с индексом и значением."""
    def __init__(self, index, value):
        self.index = index
        self.value = value

def map_func(data, data_type):
    """Преобразует элементы матрицы/вектора в пары (reducer_id, (метка, данные))."""
    if data_type == "matrix":
        return (data.col % NUM_REDUCERS, ("M", data))
    elif data_type == "vector":
        return (data.index % NUM_REDUCERS, ("V", data))

def reduce_function(reducer_id, data):
    """Умножает соответствующие элементы матрицы и вектора для заданного редьюсера."""
    matrix_parts = []
    vector_parts = []
    for tag, item in data:
        if tag == "M":
            matrix_parts.append(item)
        else:
            vector_parts.append(item)
    results = []
    for row in matrix_parts:
        for el in vector_parts:
            if row.col == el.index:
                results.append((row.row, row.value * el.value))
    return results

def map_reduce_algorithm(matrix, vector):
    """Выполняет умножение матрицы на вектор с использованием подхода MapReduce."""
    mapped_matrix = [map_func(row, "matrix") for row in matrix]
    mapped_vector = [map_func(el, "vector") for el in vector]

    grouped_data = defaultdict(list)
    for reducer_id, data_pair in mapped_matrix + mapped_vector:
        grouped_data[reducer_id].append(data_pair)

    partial_results = []
    for reducer_id, data in grouped_data.items():
        partial_results.extend(reduce_function(reducer_id, data))

    final_result = defaultdict(float)
    for row, value in partial_results:
        final_result[row] += value
    return dict(final_result)


matrix = [MatrixRow(0, 0, 1.0), MatrixRow(0, 1, 2.0), MatrixRow(1, 0, 3.0), MatrixRow(1, 1, 4.0)]
vector = [VectorElement(0, 0.5), VectorElement(1, 0.7)]

result = map_reduce_algorithm(matrix, vector)
print("Результат умножения матрицы на вектор:", result)

Matrix-Vector Multiplication Result: {0: 1.9, 1: 4.3}


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [60]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [65]:
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)  # Матрица в памяти
big_mat = np.random.rand(J,K)   # Большая матрица

def RECORDREADER():
    """Генерирует элементы большой матрицы по строкам и столбцам"""
    for j in range(big_mat.shape[0]):
        for k in range(big_mat.shape[1]):
            yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
    """Для каждого элемента большой матрицы генерирует пары для умножения"""
    j, k = k1
    w = v1
    # Для каждого элемента в столбце j малой матрицы
    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i,j] * w)  # (i,k) как ключ, произведение как значение

def REDUCE(key, values):
    """Суммирует произведения для получения итогового элемента результата"""
    i, k = key
    yield (key, sum(values))

# Реализация MapReduce
def map_reduce():
    # Этап Map
    mapped_data = []
    for key, value in RECORDREADER():
        for k2, v2 in MAP(key, value):
            mapped_data.append((k2, v2))

    # Группировка по ключам
    grouped_data = defaultdict(list)
    for key, value in mapped_data:
        grouped_data[key].append(value)

    # Этап Reduce
    result = {}
    for key, values in grouped_data.items():
        for k3, v3 in REDUCE(key, values):
            result[k3] = v3

    # Преобразование результата в матрицу
    result_mat = np.zeros((I, K))
    for (i, k), value in result.items():
        result_mat[i,k] = value

    return result_mat

# Проверка корректности
result_mapreduce = map_reduce()
result_numpy = small_mat @ big_mat  # Эталонное умножение через numpy

print("MapReduce результат:\n", result_mapreduce)
print("NumPy результат:\n", result_numpy)
print(np.allclose(result_mapreduce, result_numpy))

MapReduce результат:
 [[0.80152442 0.54008418 0.59739202 0.6088377  0.93969716 0.58997006
  0.20189569 0.60353554 0.27655655 0.68633425 0.38542162 0.70833432
  0.31472033 0.75282649 0.6492556  0.45799864 0.72009991 0.37798645
  0.59061815 0.56770832 0.74114955 0.73844761 0.15726999 0.18735378
  0.65475734 0.60735962 0.55154626 0.41039647 0.24642755 0.75777899
  0.66746044 0.21739991 0.84129279 0.84712073 0.75014048 0.39163915
  0.51205865 0.40849279 0.18817461 0.74170677]
 [0.60437843 0.34222469 0.52634528 0.46795886 0.62726674 0.35623057
  0.20952146 0.40607224 0.29801991 0.41417663 0.39689279 0.58817197
  0.24856897 0.58242372 0.57424942 0.36128614 0.46888032 0.28886717
  0.37518719 0.37847732 0.46353945 0.47772905 0.23211704 0.23412211
  0.58011841 0.43889215 0.45684616 0.41870491 0.15260445 0.48041507
  0.50354405 0.3467304  0.51545175 0.62278343 0.52052149 0.34330707
  0.44202842 0.25194753 0.30390362 0.44258762]]
NumPy результат:
 [[0.80152442 0.54008418 0.59739202 0.6088377  0.9

Проверьте своё решение

In [66]:
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [67]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [70]:
# Размеры матриц
I = 2
J = 3
K = 4

# Генерация случайных матриц
matrix_a = np.random.rand(I, J)
matrix_b = np.random.rand(J, K)

def RECORDREADER():
    """Генератор элементов матриц A и B"""
    # Элементы матрицы A
    for i in range(matrix_a.shape[0]):
        for j in range(matrix_a.shape[1]):
            yield ("A", (i, j), matrix_a[i, j])

    # Элементы матрицы B
    for j in range(matrix_b.shape[0]):
        for k in range(matrix_b.shape[1]):
            yield ("B", (j, k), matrix_b[j, k])

def MAP(key, value):
    """Функция Map для распределенного умножения матриц"""
    matrix_type, index, element = key, value[0], value[1]
    if matrix_type == "A":
        # Для элемента A[i,j] генерируем пары для всех k
        for k in range(K):
            yield ((index[0], k), ('A', index[1], element))
    elif matrix_type == "B":
        # Для элемента B[j,k] генерируем пары для всех i
        for i in range(I):
            yield ((i, index[1]), ('B', index[0], element))

def REDUCE(key, values):
    """Функция Reduce для суммирования произведений"""
    i, k = key
    # Разделяем значения A и B
    a_values = {}
    b_values = {}
    for val in values:
        if val[0] == 'A':
            a_values[val[1]] = val[2]  # val[1] = j
        else:
            b_values[val[1]] = val[2]  # val[1] = j

    # Вычисляем сумму произведений для всех j
    total = 0.0
    for j in range(J):
        if j in a_values and j in b_values:
            total += a_values[j] * b_values[j]

    yield (key, total)

# Этап Map
mapped_data = defaultdict(list)
for data in RECORDREADER():
    for intermediate_key, intermediate_value in MAP(data[0], (data[1], data[2])):
        mapped_data[intermediate_key].append(intermediate_value)

# Этап Reduce
reduced_data = {}
for key, values in mapped_data.items():
    for reduced_key, reduced_value in REDUCE(key, values):
        reduced_data[reduced_key] = reduced_value

# Формируем результирующую матрицу
result_matrix = np.zeros((I, K))
for (i, k), value in reduced_data.items():
    result_matrix[i, k] = value

print("Результат MapReduce:\n", result_matrix)

# Проверка с помощью NumPy
numpy_result = np.dot(matrix_a, matrix_b)
print("\nПроверка (NumPy):\n", numpy_result)

# Сравнение результатов
tolerance = 1e-9
print(np.allclose(result_matrix, numpy_result, atol=tolerance))

Результат MapReduce:
 [[1.22840122 0.8774328  1.39656719 1.174153  ]
 [0.74293334 0.44074656 0.85109562 0.69415147]]

Проверка (NumPy):
 [[1.22840122 0.8774328  1.39656719 1.174153  ]
 [0.74293334 0.44074656 0.85109562 0.69415147]]
True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [84]:
NUM_MAPPERS = 2
NUM_REDUCERS = 2
ROWS_A = 100
COLS_A = 50
COLS_B = 40


A = np.random.rand(ROWS_A, COLS_A)
B = np.random.rand(COLS_A, COLS_B)

def generate_chunks():
    """Генерирует диапазоны строк матрицы A для обработки мапперами."""
    chunk_size = int(np.ceil(ROWS_A / NUM_MAPPERS))
    for i_start in range(0, ROWS_A, chunk_size):
        yield (i_start, min(i_start + chunk_size, ROWS_A))


def map_function(chunk_start, chunk_end):
    """Выполняет умножение части матрицы A на матрицу B для заданного диапазона строк."""
    intermediate_results = defaultdict(list)
    for i in range(chunk_start, chunk_end):
        for j in range(COLS_A):
            for k in range(COLS_B):
                intermediate_results[(i, k)].append(A[i, j] * B[j, k])
    return intermediate_results


def partition_function(key, value):
    """Определяет редьюсер для обработки каждого промежуточного результата."""
    i, k = key
    return i % NUM_REDUCERS, (key, value)


def reduce_function(reducer_id, intermediate_results):
    """Суммирует промежуточные результаты для получения финальных значений."""
    reduced_results = {}
    for key, values in intermediate_results.items():
        reduced_results[key] = sum(values)
    return reduced_results


mapper_results = [map_function(chunk_start, chunk_end) for chunk_start, chunk_end in generate_chunks()]


partitioned_data = defaultdict(list)
for mapper_output in mapper_results:
    for key, value in mapper_output.items():
        reducer_id, data_item = partition_function(key, value)
        partitioned_data[reducer_id].append(data_item)


reducer_results = [reduce_function(reducer_id, dict(partitioned_data[reducer_id])) for reducer_id in partitioned_data]

final_result = {}
for reducer_output in reducer_results:
    final_result.update(reducer_output)

print("Matrix Multiplication Result (partial):\n", dict(final_result))

result_matrix = np.zeros((ROWS_A, COLS_B))
for (i, k), value in final_result.items():
    result_matrix[i, k] = value


print("\nResult Matrix (NumPy):\n", result_matrix)

numpy_result = np.dot(A, B)
print("\nVerification (NumPy):\n", numpy_result)

tolerance = 1e-9
print(np.allclose(result_matrix, numpy_result, atol=tolerance))



Matrix Multiplication Result (partial):
 {(0, 0): np.float64(12.687360278165391), (0, 1): np.float64(12.837756019682967), (0, 2): np.float64(13.287790442753252), (0, 3): np.float64(13.208463956958859), (0, 4): np.float64(14.339404325958965), (0, 5): np.float64(13.548276492913946), (0, 6): np.float64(11.91753578332218), (0, 7): np.float64(13.302353257329024), (0, 8): np.float64(12.78878194277444), (0, 9): np.float64(14.367237201858774), (0, 10): np.float64(12.058510695331487), (0, 11): np.float64(12.980210313395737), (0, 12): np.float64(13.809928780520092), (0, 13): np.float64(13.246360961420145), (0, 14): np.float64(12.082672316270767), (0, 15): np.float64(15.150597132627494), (0, 16): np.float64(13.431076759077728), (0, 17): np.float64(11.892231237479507), (0, 18): np.float64(13.801487272916496), (0, 19): np.float64(13.82791239799298), (0, 20): np.float64(13.169804354424421), (0, 21): np.float64(14.324232315284952), (0, 22): np.float64(14.234428241218955), (0, 23): np.float64(14.06998

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [88]:
I, J, K = 4, 5, 80  # Размеры матриц

small_mat = np.random.rand(I,J) # Малая матрица
big_mat = np.random.rand(J,K) # Большая матрица
reference_solution = np.matmul(small_mat, big_mat) # Эталонное решение

def INPUTFORMAT():
  """Генератор чанков данных для матриц"""
  first_mat = []
  for i in range(small_mat.shape[0]):
    for j in range(small_mat.shape[1]):
      first_mat.append(((0, i, j), small_mat[i,j]))

  global maps
  split_size =  int(np.ceil(len(first_mat)/maps))
  for i in range(0, len(first_mat), split_size):
    yield first_mat[i:i+split_size]

  second_mat = []
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      second_mat.append(((1, j, k), big_mat[j,k]))

  split_size =  int(np.ceil(len(second_mat)/maps))
  for i in range(0, len(second_mat), split_size):
    yield second_mat[i:i+split_size]


def MAP_JOIN(k1, v1):
  """Маппер для этапа соединения матриц по общему индексу"""
  (mat_num, i, j) = k1
  w = v1
  if mat_num == 0:
    yield (j, (mat_num, i, w))
  else:
    yield (i, (mat_num, j, w))


def REDUCE_JOIN(key, values):
  from_first_mat = [v for v in values if v[0] == 0]
  from_second_mat = [v for v in values if v[0] == 1]
  for f in from_first_mat:
    for s in from_second_mat:
      yield ((f[1], s[1]), f[2] * s[2])


def GET_JOINED():
  """Генератор соединенных данных для этапа умножения"""
  for j in joined:
    print("aa", j)
    yield j[1]


def MAP_MUL(k1, v1):
  """Маппер для этапа умножения (передает данные)"""
  yield (k1, v1)


def REDUCE_MUL(key, values):
  """Редьюсер для суммирования результатов умножения"""
  res_val = 0
  for v in values:
    res_val += v
  yield (key, res_val)

maps = 3
reducers = 2
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP_JOIN, REDUCE_JOIN, COMBINER=None)
joined = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]

mul_output = MapReduceDistributed(GET_JOINED, MAP_MUL, REDUCE_MUL, COMBINER=None)
pre_result = [(partition_id, list(partition)) for (partition_id, partition) in mul_output]

solution = []
for p in pre_result:
  for v in p[1]:
    solution.append(v)

print(solution)
np.allclose(reference_solution, asmatrix(solution)) # should return true

420 key-value pairs were sent over a network.
aa (0, [((0, 0), np.float64(0.009110929692112982)), ((0, 1), np.float64(0.1896277281136618)), ((0, 2), np.float64(0.26700125401426716)), ((0, 3), np.float64(0.1742642301302729)), ((0, 4), np.float64(0.6317911449414206)), ((0, 5), np.float64(0.7024655987529195)), ((0, 6), np.float64(0.1294958754433979)), ((0, 7), np.float64(0.2747534328675044)), ((0, 8), np.float64(0.7487198985784205)), ((0, 9), np.float64(0.06909126247687633)), ((0, 10), np.float64(0.34417647433447446)), ((0, 11), np.float64(0.5237170111603693)), ((0, 12), np.float64(0.08849895151827607)), ((0, 13), np.float64(0.06272538493397133)), ((0, 14), np.float64(0.6835342422802281)), ((0, 15), np.float64(0.5875412747615858)), ((0, 16), np.float64(0.3026070232863796)), ((0, 17), np.float64(0.2065773819386114)), ((0, 18), np.float64(0.14149981861192)), ((0, 19), np.float64(0.40681859850107605)), ((0, 20), np.float64(0.732936861291247)), ((0, 21), np.float64(0.43132976791855104)), ((0,

True